In [1]:
import numpy as np
import jax
import jax.numpy as jnp
from functools import partial
from time import time

In [3]:
def attn(X, indices, gamma):
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    Y = np.empty_like(X)
    
    one_minus_gamma = 1.0 - gamma
    
    # Проходим по каждому сегменту, заданному границами
    for i in range(len(indices) - 1):
        start, end = indices[i], indices[i+1]
        if start == end:
            continue
            
        segment = X[start:end]
        n = len(segment)
        
        if gamma == 0:
            Y[start:end] = segment
        else:
            # Векторизованное вычисление EMA:
            # Y[t] = (1-gamma) * gamma^t * cumsum(X[t] * gamma^-t)
            powers = gamma ** np.arange(n)
            weighted = segment / powers
            cumweighted = np.cumsum(weighted)
            Y[start:end] = one_minus_gamma * powers * cumweighted
            
    return Y

In [4]:
def attn_vectorized(X, indices, gamma):
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    I = len(X)
    Y = np.empty_like(X)

    if gamma == 0:
        return X.copy()
    
    # Глобальные степени гаммы
    arange = np.arange(I)
    powers = gamma ** arange
    powers_inv = 1.0 / powers  # gamma ** -arange
    
    # Взвешенные значения для кумулятивной суммы
    weighted = X * powers_inv
    global_cum = np.cumsum(weighted)
    
    # Векторизованная сегментированная кумулятивная сумма
    # 1. Создаем маску сброса на началах сегментов (кроме первого)
    resets = indices[1:-1]
    mask = np.zeros(I, dtype=int)
    if len(resets) > 0:
        mask[resets] = resets
        
    # 2. Для каждой позиции находим индекс последнего сброса
    last_reset = np.maximum.accumulate(mask)
    
    # 3. Вычисляем величину коррекции (значение глобальной суммы перед сбросом)
    correction = np.zeros(I)
    valid = last_reset > 0
    correction[valid] = global_cum[last_reset[valid] - 1]
    
    # 4. Сегментированная сумма
    seg_cum = global_cum - correction
    
    # Финальный расчет EMA
    Y = (1.0 - gamma) * powers * seg_cum
    return Y

In [19]:
X = np.arange(20) + 1
Y = np.arange(20) + 2
Z = np.vstack((X, Y))
indices = [0, 20]
gamma = 0.1
beta = 0.5
Z

array([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20],
       [ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21]])

In [11]:
attn(X, indices, gamma)

array([4.        , 6.4       , 7.84      , 4.        , 6.4       ,
       7.84      , 8.704     , 9.2224    , 9.53344   , 9.720064  ,
       4.        , 6.4       , 7.84      , 8.704     , 9.2224    ,
       9.53344   , 9.720064  , 9.8320384 , 9.89922304, 9.93953382])

In [12]:
attn_vectorized(X, indices, gamma)

array([4.        , 6.4       , 7.84      , 4.        , 6.4       ,
       7.84      , 8.704     , 9.2224    , 9.53344   , 9.720064  ,
       4.        , 6.4       , 7.84      , 8.704     , 9.2224    ,
       9.53344   , 9.720064  , 9.8320384 , 9.89922304, 9.93953382])

In [1]:
def attn(X, indices, gamma):
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    I = len(X)
    
    if gamma == 1.0:
        return X.copy()
    
    beta = 1.0 - gamma
    arange = np.arange(I)
    
    # P[t] = beta^t, P_inv[t] = beta^-t
    P = beta ** arange
    P_inv = 1.0 / P
    
    # Global weighted cumsum
    W = X * P_inv
    global_cum = np.cumsum(W)
    
    # Segmented cumsum via correction mask
    resets = indices[1:-1]
    mask = np.zeros(I, dtype=int)
    if len(resets) > 0:
        mask[resets] = resets
        
    last_reset = np.maximum.accumulate(mask)
    correction = np.zeros(I)
    valid = last_reset > 0
    correction[valid] = global_cum[last_reset[valid] - 1]
    
    seg_cum = global_cum - correction
    
    # Broadcast segment start values
    lengths = np.diff(indices)
    X_start = np.repeat(X[indices[:-1]], lengths)
    P_start_inv = np.repeat(P_inv[indices[:-1]], lengths)
    
    # Formula derived from recurrence: y_t = beta^t * (beta * x_start * beta^-start + gamma * seg_cum_t)
    Y = P * (beta * X_start * P_start_inv + gamma * seg_cum)
    
    return Y

In [9]:
attn(X, indices, gamma)

array([ 1.        ,  1.1       ,  1.29      ,  1.561     ,  1.9049    ,
        2.31441   ,  2.782969  ,  3.3046721 ,  3.87420489,  4.4867844 ,
       11.        , 11.1       , 11.29      , 11.561     , 11.9049    ,
       12.31441   , 12.782969  , 13.3046721 , 13.87420489, 14.4867844 ])

In [10]:
def attn(X, indices, gamma, beta):
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    I = len(X)
    
    if gamma == 1.0:
        return X.copy()
    
    alpha = 1.0 - gamma
    arange = np.arange(I)
    alpha_pow = alpha ** arange
    alpha_inv_pow = 1.0 / alpha_pow

    def calc_right_ema(X_in, idx_in):
        W = X_in * alpha_inv_pow
        global_cum = np.cumsum(W)
        
        resets = idx_in[1:-1]
        mask = np.zeros(I, dtype=int)
        if len(resets) > 0:
            mask[resets] = resets
            
        last_reset = np.maximum.accumulate(mask)
        correction = np.zeros(I)
        valid = last_reset > 0
        correction[valid] = global_cum[last_reset[valid] - 1]
        
        seg_cum = global_cum - correction
        
        lengths = np.diff(idx_in)
        starts = idx_in[:-1]
        X_starts = np.repeat(X_in[starts], lengths)
        alpha_inv_starts = np.repeat(alpha_inv_pow[starts], lengths)
        
        term1 = alpha_pow * alpha * alpha_inv_starts * X_starts
        term2 = gamma * alpha_pow * seg_cum
        return term1 + term2

    Y = calc_right_ema(X, indices)
    
    X_rev = X[::-1]
    rev_indices = I - indices[::-1]
    Z_rev = calc_right_ema(X_rev, rev_indices)
    Z = Z_rev[::-1]
    
    return beta * Y + (1.0 - beta) * Z

In [35]:
print(attn(X, indices, gamma, beta=0.5))
print(attn(Y, indices, gamma, beta=0.5))
print(attn_2d(Z, indices, gamma, beta=0.5))
Z

[ 4.89211673  5.37457414  5.89452682  6.44664092  7.02593991  7.62774934
  8.24764488  8.88140314  9.52495476 10.17433922 10.82566078 11.47504524
 12.11859686 12.75235512 13.37225066 13.97406009 14.55335908 15.10547318
 15.62542586 16.10788327]
[ 5.89211673  6.37457414  6.89452682  7.44664092  8.02593991  8.62774934
  9.24764488  9.88140314 10.52495476 11.17433922 11.82566078 12.47504524
 13.11859686 13.75235512 14.37225066 14.97406009 15.55335908 16.10547318
 16.62542586 17.10788327]
[[ 4.89211673  5.37457414  5.89452682  6.44664092  7.02593991  7.62774934
   8.24764488  8.88140314  9.52495476 10.17433922 10.82566078 11.47504524
  12.11859686 12.75235512 13.37225066 13.97406009 14.55335908 15.10547318
  15.62542586 16.10788327]
 [ 4.89211673  5.37457414  5.89452682  6.44664092  7.02593991  7.62774934
   8.24764488  8.88140314  9.52495476 10.17433922 10.82566078 11.47504524
  12.11859686 12.75235512 13.37225066 13.97406009 14.55335908 15.10547318
  15.62542586 16.10788327]]


array([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20],
       [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20]])

In [131]:
def bidir_ema(X, 
            indices, 
            gamma: float, 
            beta: float = 0.5
        ) -> np.array:
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    H, I = X.shape
    
    if gamma == 1.0:
        return X.copy()
    
    alpha = 1.0 - gamma
    arange = np.arange(I)
    alpha_pow = alpha ** arange
    alpha_inv_pow = 1.0 / alpha_pow

    def calc_right_ema(X_in, idx_in):
        W = X_in * alpha_inv_pow
        global_cum = np.cumsum(W, axis=1)
        
        resets = idx_in[1:-1]
        mask = np.zeros(I, dtype=int)
        if len(resets) > 0:
            mask[resets] = resets
            
        last_reset = np.maximum.accumulate(mask)
        correction = np.zeros((H, I))
        valid = last_reset > 0
        if np.any(valid):
            correction[:, valid] = global_cum[:, last_reset[valid] - 1]
        
        seg_cum = global_cum - correction
        
        lengths = np.diff(idx_in)
        starts = idx_in[:-1]
        X_starts = np.repeat(X_in[:, starts], lengths, axis=1)
        alpha_inv_starts = np.repeat(alpha_inv_pow[starts], lengths)
        
        term1 = alpha_pow * alpha * alpha_inv_starts * X_starts
        term2 = gamma * alpha_pow * seg_cum
        return term1 + term2

    Y = calc_right_ema(X, indices)
    
    X_rev = X[:, ::-1]
    rev_indices = I - indices[::-1]
    left_ema_rev = calc_right_ema(X_rev, rev_indices)
    left_ema = left_ema_rev[:, ::-1]
    
    return beta * Y + (1.0 - beta) * left_ema

print(bidir_ema(Z, indices, gamma))

[[   5.5      6.05     6.645 ... 4994.355 4994.95  4995.5  ]
 [   6.5      7.05     7.645 ... 4995.355 4995.95  4996.5  ]]


In [133]:
def bidir_ema(X, 
              indices, 
              gamma, 
              beta=0.5
             ) -> np.array:
    
    X = np.asarray(X, dtype=float)
    indices = np.asarray(indices, dtype=int)
    H, I = X.shape
    
    if gamma == 1.0:
        return X.copy()
    
    alpha = 1.0 - gamma
    arange = np.arange(I)
    alpha_pow = alpha ** arange
    alpha_inv_pow = 1.0 / alpha_pow

    def calc_right_ema(X_in, idx_in):
        W = X_in * alpha_inv_pow
        global_cum = np.cumsum(W, axis=1)
        
        resets = idx_in[1:-1]
        mask = np.zeros(I, dtype=int)
        if len(resets) > 0:
            mask[resets] = resets
            
        last_reset = np.maximum.accumulate(mask)
        correction = np.zeros((H, I))
        valid = last_reset > 0
        if np.any(valid):
            correction[:, valid] = global_cum[:, last_reset[valid] - 1]
        
        seg_cum = global_cum - correction
        
        lengths = np.diff(idx_in)
        starts = idx_in[:-1]
        X_starts = np.repeat(X_in[:, starts], lengths, axis=1)
        alpha_inv_starts = np.repeat(alpha_inv_pow[starts], lengths)
        
        term1 = alpha_pow * alpha * alpha_inv_starts * X_starts
        term2 = gamma * alpha_pow * seg_cum
        return term1 + term2

    right_ema = calc_right_ema(X, indices)
    
    X_rev = X[:, ::-1]
    rev_indices = I - indices[::-1]
    left_ema_rev = calc_right_ema(X_rev, rev_indices)
    left_ema = left_ema_rev[:, ::-1]
    
    return beta * right_ema + (1.0 - beta) * left_ema

print(bidir_ema(Z, indices, gamma))

[[   5.5      6.05     6.645 ... 4994.355 4994.95  4995.5  ]
 [   6.5      7.05     7.645 ... 4995.355 4995.95  4996.5  ]]


In [117]:
print(bidir_ema(Z, indices, gamma, beta=0.5))

[[   5.5      6.05     6.645 ... 4994.355 4994.95  4995.5  ]
 [   6.5      7.05     7.645 ... 4995.355 4995.95  4996.5  ]]


In [69]:
Z = np.vstack((X, Y))
print(attn_2d(Z, indices, gamma, beta=0.5))

[[ 4.89211673  5.37457414  5.89452682  6.44664092  7.02593991  7.62774934
   8.24764488  8.88140314  9.52495476 10.17433922 10.82566078 11.47504524
  12.11859686 12.75235512 13.37225066 13.97406009 14.55335908 15.10547318
  15.62542586 16.10788327]
 [ 5.89211673  6.37457414  6.89452682  7.44664092  8.02593991  8.62774934
   9.24764488  9.88140314 10.52495476 11.17433922 11.82566078 12.47504524
  13.11859686 13.75235512 14.37225066 14.97406009 15.55335908 16.10547318
  16.62542586 17.10788327]]


In [77]:
from functools import partial
import jax
import jax.numpy as jnp

@partial(jax.jit, static_argnames=['gamma', 'beta'])
def attn_2d_jax(X: jax.Array, indices: jax.Array, gamma: float, beta: float) -> jax.Array:
    X = jnp.asarray(X)
    indices = jnp.asarray(indices, dtype=int)
    H, I = X.shape
    
    if gamma == 1.0:
        return X.copy()
    
    alpha = 1.0 - gamma
    alpha_pow = alpha ** jnp.arange(I)
    alpha_inv_pow = 1.0 / alpha_pow

    def calc_right_ema(X_in: jax.Array, idx_in: jax.Array) -> jax.Array:
        W = X_in * alpha_inv_pow
        global_cum = jnp.cumsum(W, axis=1)
        
        resets = jnp.asarray(idx_in[1:-1], dtype=int).ravel()
        
        # Маска сброса через scatter (resets — гарантированно jax.Array)
        mask = jnp.zeros(I, dtype=int).at[resets].set(resets)
        last_reset = jnp.maximum.accumulate(mask)
        
        # Коррекция кумулятивной суммы
        valid = last_reset > 0
        safe_idx = jnp.where(valid, last_reset - 1, 0)
        correction_vals = global_cum[:, safe_idx]
        correction = jnp.where(valid[None, :], correction_vals, 0)
        
        seg_cum = global_cum - correction
        
        lengths = jnp.diff(idx_in)
        starts = idx_in[:-1]
        
        # Ключевое исправление: указываем total_repeat_length=I
        X_starts = jnp.repeat(X_in[:, starts], lengths, axis=1, total_repeat_length=I)
        alpha_inv_starts = jnp.repeat(alpha_inv_pow[starts], lengths, total_repeat_length=I)
        
        term1 = alpha_pow * alpha * alpha_inv_starts * X_starts
        term2 = gamma * alpha_pow * seg_cum
        return term1 + term2

    Y = calc_right_ema(X, indices)
    
    X_rev = X[:, ::-1]
    rev_indices = I - indices[::-1]
    Z_rev = calc_right_ema(X_rev, rev_indices)
    Z = Z_rev[:, ::-1]
    
    return beta * Y + (1.0 - beta) * Z

In [78]:
print(attn_2d_jax(Z, indices, gamma, beta=0.5))

[[ 4.8921156  5.3745728  5.8945255  6.4466395  7.025938   7.627748
   8.2476425  8.881402   9.524953  10.174337  10.82566   11.475044
  12.118597  12.752354  13.37225   13.974058  14.553358  15.105472
  15.625425  16.107883 ]
 [ 5.892115   6.3745728  6.894525   7.4466395  8.025937   8.627748
   9.2476425  9.881402  10.524953  11.174337  11.825659  12.475044
  13.118595  13.752354  14.37225   14.974058  15.553358  16.105473
  16.625423  17.107883 ]]


In [100]:
n = 5000
X = np.arange(n) + 1;
Y = np.arange(n) + 2;
Z = np.vstack((X, Y));
indices = [0, n/2, n];
gamma = 0.1;
beta = 0.5

In [106]:
print(jax.devices())
start_time = time()
for i in range(50):
    a = attn_2d_jax(Z + i, indices, gamma=0.1, beta=0.5)
print(time() - start_time)

[CudaDevice(id=0)]
16.44405722618103


In [104]:
start_time = time()
for i in range(50):
    a = attn_2d(Z + i, indices, gamma, beta=0.5)
print(time() - start_time)

0.021513938903808594


In [86]:
Z

array([[    1,     2,     3, ..., 19998, 19999, 20000],
       [    2,     3,     4, ..., 19999, 20000, 20001]], shape=(2, 20000))

In [7]:
from jax import Array
x = jnp.array([0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,
0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,
0.,	0.,	0.,	0.,	0.,	0.,	1.,	0.,	1.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,
1.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,
0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,
0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.,	0.])
x = jnp.vstack([x, x])
x

Array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0.]], dtype=float32)

In [8]:
@jax.jit
def _process_row(x: Array, indices: Array, gamma: float, beta: float) -> Array:
    alpha = 1.0 - gamma
    seq_len = x.shape[0]
    
    # Маска сброса для прямого прохода (начало сегмента)
    is_start = jnp.zeros(seq_len, dtype=bool).at[indices[:-1]].set(True)
    # Маска сброса для обратного прохода (конец сегмента)
    is_end   = jnp.zeros(seq_len, dtype=bool).at[indices[1:] - 1].set(True)
    
    # Общий шаг скана
    def scan_step(carry, inputs):
        val, is_reset = inputs
        # Если сброс -> новое значение = x, иначе рекуррентная формула
        new_val = jnp.where(is_reset, val, gamma * val + alpha * carry)
        return new_val, new_val
        
    # Прямой EMA (сброс на is_start)
    _, y = jax.lax.scan(scan_step, 0.0, (x, is_start))
    
    # Обратный EMA (переворачиваем массив и маску конца)
    x_rev = jnp.flip(x)
    is_end_rev = jnp.flip(is_end)  # Теперь True стоит на первом элементе перевёрнутого сегмента
    _, z_rev = jax.lax.scan(scan_step, 0.0, (x_rev, is_end_rev))
    z = jnp.flip(z_rev)
    
    return beta * y + (1.0 - beta) * z
    
bidir_ema_jax = jax.vmap(_process_row, in_axes=(0, None, None, None))

In [10]:
bidir_ema_jax(x, jnp.array([0, 97]), 0.6, 0.5)

Array([[4.20720305e-17, 1.05180085e-16, 2.62950235e-16, 6.57375600e-16,
        1.64343916e-15, 4.10859811e-15, 1.02714959e-14, 2.56787406e-14,
        6.41968575e-14, 1.60492157e-13, 4.01230427e-13, 1.00307609e-12,
        2.50769045e-12, 6.26922646e-12, 1.56730670e-11, 3.91826710e-11,
        9.79566844e-11, 2.44891718e-10, 6.12229323e-10, 1.53057345e-09,
        3.82643384e-09, 9.56608570e-09, 2.39152165e-08, 5.97880430e-08,
        1.49470111e-07, 3.73675306e-07, 9.34188336e-07, 2.33547098e-06,
        5.83867768e-06, 1.45966951e-05, 3.64917396e-05, 9.12293544e-05,
        2.28073404e-04, 5.70183562e-04, 1.42545905e-03, 3.56364786e-03,
        8.90912022e-03, 2.22728010e-02, 5.56820072e-02, 1.39205024e-01,
        6.48012638e-01, 2.40031451e-01, 6.48078680e-01, 1.39396608e-01,
        5.61715178e-02, 2.35007983e-02, 1.19807981e-02, 1.12435175e-02,
        2.06254050e-02, 4.85701598e-02, 1.20228060e-01, 6.00091219e-01,
        1.20036490e-01, 4.80145924e-02, 1.92058366e-02, 7.682334

In [12]:
@partial(jax.jit)
def _ema_windowed_attn(
        *,
        x: jax.Array,
        ctx_bounds: jax.Array,
) -> jax.Array:
    if x.shape[1] == 0:
        return x

    ctx_len = 200
    gamma_i = 0.6
    gamma_n = 0.6
    beta = 0.5
    seq_len = x.shape[1] #// для p_ti.T это I
    doc_ids = jnp.searchsorted(ctx_bounds[1:], jnp.arange(seq_len), side='right') #// массив номеров документов для каждой позиции
    doc_starts = jnp.concatenate([ #// позиции начала документа
        jnp.array([True], dtype=bool),
        doc_ids[1:] != doc_ids[:-1],
    ])
    doc_ends = jnp.concatenate([ #// позиции за концом документа
        doc_ids[:-1] != doc_ids[1:],
        jnp.array([True], dtype=bool),
    ])

    forward = jnp.zeros_like(x)
    backward = jnp.zeros_like(x)

    # Ограничиваем влияние соседей окном длины ctx_len с каждой стороны.
    # При достаточном окне формула совпадает с EMA внутри документа.
    max_offset = ctx_len if ctx_len > 0 else seq_len - 1

    for offset in range(max_offset + 1):
        if seq_len <= offset:
            break

        decay_forward = (1.0 - gamma_i) ** offset
        decay_backward = (1.0 - gamma_n) ** offset

        if offset == 0:
            same_doc = jnp.ones((seq_len,), dtype=x.dtype)
            src_forward = x
            src_backward = x
            coeff_forward = decay_forward * (
                    gamma_i + (1.0 - gamma_i) * doc_starts.astype(x.dtype)
            )
            coeff_backward = decay_backward * (
                    gamma_n + (1.0 - gamma_n) * doc_ends.astype(x.dtype)
            )
            forward = forward + src_forward * coeff_forward[None, :]
            backward = backward + src_backward * coeff_backward[None, :]
            continue

        same_doc = (doc_ids[offset:] == doc_ids[:-offset]).astype(x.dtype)

        src_forward = x[:, :-offset]
        coeff_forward = decay_forward * (
                gamma_i + (1.0 - gamma_i) * doc_starts[:-offset].astype(x.dtype)
        )
        forward = forward.at[:, offset:].add(
            src_forward * (coeff_forward * same_doc)[None, :]
        )

        src_backward = x[:, offset:]
        coeff_backward = decay_backward * (
                gamma_n + (1.0 - gamma_n) * doc_ends[offset:].astype(x.dtype)
        )
        backward = backward.at[:, :-offset].add(
            src_backward * (coeff_backward * same_doc)[None, :]
        )

    return beta * forward + (1.0 - beta) * backward


In [14]:
_ema_windowed_attn(x=x, ctx_bounds=jnp.array([0, 97]))

E0430 18:06:38.843570   73882 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0430 18:06:39.183539   73871 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0430 18:06:39.291569   73883 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0430 18:06:39.398811   73877 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0430 18:06:39.496524   73880 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0430 18:06:39.604179   73871 cuda_timer.cc:8

Array([[4.2072140e-17, 1.0518036e-16, 2.6295090e-16, 6.5737719e-16,
        1.6434431e-15, 4.1086078e-15, 1.0271519e-14, 2.5678798e-14,
        6.4197000e-14, 1.6049247e-13, 4.0123119e-13, 1.0030780e-12,
        2.5076950e-12, 6.2692377e-12, 1.5673095e-11, 3.9182733e-11,
        9.7956830e-11, 2.4489211e-10, 6.1223027e-10, 1.5305754e-09,
        3.8264392e-09, 9.5660964e-09, 2.3915245e-08, 5.9788107e-08,
        1.4947027e-07, 3.7367568e-07, 9.3418919e-07, 2.3354728e-06,
        5.8386822e-06, 1.4596706e-05, 3.6491769e-05, 9.1229405e-05,
        2.2807354e-04, 5.7018385e-04, 1.4254596e-03, 3.5636488e-03,
        8.9091230e-03, 2.2272807e-02, 5.5682011e-02, 1.3920504e-01,
        6.4801264e-01, 2.4003147e-01, 6.4807868e-01, 1.3939661e-01,
        5.6171518e-02, 2.3500802e-02, 1.1980801e-02, 1.1243520e-02,
        2.0625409e-02, 4.8570164e-02, 1.2022807e-01, 6.0009122e-01,
        1.2003650e-01, 4.8014596e-02, 1.9205838e-02, 7.6823356e-03,
        3.0729342e-03, 1.2291737e-03, 4.9166952e

In [4]:
n_tw = jnp.arange(12).reshape((3,4))
n_tw

Array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]], dtype=int32)

In [6]:
p_ti = jnp.arange(15).reshape((3,5))
p_ti

Array([[ 0,  1,  2,  3,  4],
       [ 5,  6,  7,  8,  9],
       [10, 11, 12, 13, 14]], dtype=int32)

In [13]:
batch = jnp.array([1, 2, 1, 3, 2])
batch

Array([1, 2, 1, 3, 2], dtype=int32)

In [14]:
rows = jnp.arange(3)[:, None]
rows

Array([[0],
       [1],
       [2]], dtype=int32)

In [15]:
jnp.add.at(n_tw, (rows, batch), p_ti, inplace=False)

Array([[ 0,  3,  7,  6],
       [ 4, 17, 21, 15],
       [ 8, 31, 35, 24]], dtype=int32)

In [20]:
n_tw.at[:, batch].add(p_ti)

Array([[ 0,  3,  7,  6],
       [ 4, 17, 21, 15],
       [ 8, 31, 35, 24]], dtype=int32)

In [19]:
H, W = 3, 4 
arr = jnp.zeros((H, W))
row_idx = jnp.arange(H)          # [0, 1, 2, ...]
col_idx = jnp.array([1, 3, 2])   # целевые колонки
vals  = jnp.array([1.0, 2.0, 3.0])

arr = arr.at[row_idx, col_idx].add(vals)
arr

Array([[0., 1., 0., 0.],
       [0., 0., 0., 2.],
       [0., 0., 3., 0.]], dtype=float32)